# Class 8 - Operations & Broadcasting
### Data Analytics in Python | Week 3, Saturday

Yesterday you learned how to create and index arrays. Today you learn to
**compute on them** - and to do so across arrays of different shapes without
writing a single loop.

Broadcasting is NumPy's system for aligning arrays of different shapes so
operations can still be applied. Once you understand it, code that looks
like it shouldn't work starts making perfect sense.

**Today:**
- Element-wise arithmetic and comparison
- Universal functions (ufuncs)
- Aggregation with `axis` - the most misunderstood parameter
- Broadcasting rules
- Matrix operations: dot product, transpose, norm
- Reshaping and stacking
- Practical: Vectorized BMI and z-score calculator

## 1. Element-wise arithmetic

When two arrays have the same shape, every arithmetic operator works element by element - no loop, no iteration.

In [1]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print(f"a     = {a}")
print(f"b     = {b}")
print()
print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}")
print(f"b / a = {b / a}")
print(f"b // a= {b // a}")    # floor division
print(f"a ** 2= {a ** 2}")
print(f"b % 3 = {b % 3}")

# Comparison operators return boolean arrays
print(f"\na > 3  = {a > 3}")
print(f"a == 3 = {a == 3}")
print(f"a != 3 = {a != 3}")

a     = [1 2 3 4 5]
b     = [10 20 30 40 50]

a + b = [11 22 33 44 55]
a - b = [ -9 -18 -27 -36 -45]
a * b = [ 10  40  90 160 250]
b / a = [10. 10. 10. 10. 10.]
b // a= [10 10 10 10 10]
a ** 2= [ 1  4  9 16 25]
b % 3 = [1 2 0 1 2]

a > 3  = [False False False  True  True]
a == 3 = [False False  True False False]
a != 3 = [ True  True False  True  True]


In [3]:
#why element-wise matters: computing patient delta vitals
baseline = np.array([5.0,  120, 72,  13.5])   # glucose, bp, hr, hb
current  = np.array([9.4,  145, 92,  11.8])

markers  = ["Glucose", "BP_sys", "HR", "Hb"]
delta    = current - baseline
pct_chg  = (delta / baseline) * 100

print(f"{'Marker':<10} {'Baseline':>10} {'Current':>9} {'Delta':>8} {'Change%':>9}")
print("-" * 50)
for m, b, c, d, p in zip(markers, baseline, current, delta, pct_chg):
    flag = " <--" if abs(p) > 10 else ""
    print(f"{m:<10} {b:>10.1f} {c:>9.1f} {d:>+8.1f} {p:>8.1f}%{flag}")

Marker       Baseline   Current    Delta   Change%
--------------------------------------------------
Glucose           5.0       9.4     +4.4     88.0% <--
BP_sys          120.0     145.0    +25.0     20.8% <--
HR               72.0      92.0    +20.0     27.8% <--
Hb               13.5      11.8     -1.7    -12.6% <--


## 2. Universal functions - ufuncs

Ufuncs are element-wise mathematical functions that operate directly on arrays at C speed. No loop needed.

In [4]:
data = np.array([0.25, 1.0, 4.0, 9.0, 16.0, 25.0])

print(f"Original:    {data}")
print(f"np.sqrt():   {np.sqrt(data)}")
print(f"np.log():    {np.log(data).round(4)}")        # natural log
print(f"np.log10():  {np.log10(data).round(4)}")
print(f"np.log2():   {np.log2(data).round(4)}")
print(f"np.exp():    {np.exp(np.array([0,1,2]))}")    # e^x

angles = np.linspace(0, np.pi, 5)
print(f"\nAngles (radians): {angles.round(4)}")
print(f"np.sin():    {np.sin(angles).round(4)}")
print(f"np.cos():    {np.cos(angles).round(4)}")

Original:    [ 0.25  1.    4.    9.   16.   25.  ]
np.sqrt():   [0.5 1.  2.  3.  4.  5. ]
np.log():    [-1.3863  0.      1.3863  2.1972  2.7726  3.2189]
np.log10():  [-0.6021  0.      0.6021  0.9542  1.2041  1.3979]
np.log2():   [-2.      0.      2.      3.1699  4.      4.6439]
np.exp():    [1.         2.71828183 7.3890561 ]

Angles (radians): [0.     0.7854 1.5708 2.3562 3.1416]
np.sin():    [0.     0.7071 1.     0.7071 0.    ]
np.cos():    [ 1.      0.7071  0.     -0.7071 -1.    ]


In [5]:
#practical ufuncs for data analytics
scores = np.array([0.0, 25.0, 50.0, 75.0, 100.0])

#clipping - cap values at boundaries
clipped = np.clip(scores, 10, 90)
print(f"Original: {scores}")
print(f"Clipped [10,90]: {clipped}")

#Absolute value
deltas = np.array([-5.2, 3.1, -8.7, 0.4, -1.2])
print(f"\nDeltas:    {deltas}")
print(f"abs:       {np.abs(deltas)}")
print(f"sign:      {np.sign(deltas)}")   # -1, 0, or +1

#Round, floor, ceil
values = np.array([1.2, 2.7, -1.2, -2.7, 3.5])
print(f"\nValues:    {values}")
print(f"round:     {np.round(values)}")
print(f"floor:     {np.floor(values)}")
print(f"ceil:      {np.ceil(values)}")

Original: [  0.  25.  50.  75. 100.]
Clipped [10,90]: [10. 25. 50. 75. 90.]

Deltas:    [-5.2  3.1 -8.7  0.4 -1.2]
abs:       [5.2 3.1 8.7 0.4 1.2]
sign:      [-1.  1. -1.  1. -1.]

Values:    [ 1.2  2.7 -1.2 -2.7  3.5]
round:     [ 1.  3. -1. -3.  4.]
floor:     [ 1.  2. -2. -3.  3.]
ceil:      [ 2.  3. -1. -2.  4.]


## 3. Aggregation with axis - the most misunderstood parameter

The `axis` parameter tells NumPy **which dimension to collapse**. Most students get this backwards the first time.

- `axis=0` --> collapse **rows** --> result has shape `(cols,)` --> one value per column (column wise mean)
- `axis=1` --> collapse **columns** --> result has shape `(rows,)` --> one value per row (row wise mean)

In [6]:
# 3 patients (rows), 4 markers (cols): WBC, RBC, Hb, Glucose
labs = np.array([
    [8.2,  4.9, 13.5,  5.1],   # Patient 1
    [12.1, 4.2, 11.8,  9.4],   # Patient 2
    [6.8,  5.1, 14.2,  4.8],   # Patient 3
])
markers  = ["WBC", "RBC", "Hb", "Glucose"]
patients = ["P1", "P2", "P3"]

print("Lab matrix (3 patients x 4 markers):")
print(labs)
print()

# axis=0: collapse ROWS --> result shape (4,) --> one value per COLUMN (per marker)
col_means = labs.mean(axis=0)
print("axis=0 (mean per MARKER - collapse rows):")
for m, v in zip(markers, col_means):
    print(f"  {m:<10} mean = {v:.2f}")

# axis=1: collapse COLUMNS --> result shape (3,) --> one value per ROW (per patient)
row_means = labs.mean(axis=1)
print("\naxis=1 (mean per PATIENT - collapse cols):")
for p, v in zip(patients, row_means):
    print(f"  {p}  mean across all markers = {v:.2f}")

# No axis: single scalar
print(f"\nOverall mean: {labs.mean():.2f}")

Lab matrix (3 patients x 4 markers):
[[ 8.2  4.9 13.5  5.1]
 [12.1  4.2 11.8  9.4]
 [ 6.8  5.1 14.2  4.8]]

axis=0 (mean per MARKER - collapse rows):
  WBC        mean = 9.03
  RBC        mean = 4.73
  Hb         mean = 13.17
  Glucose    mean = 6.43

axis=1 (mean per PATIENT - collapse cols):
  P1  mean across all markers = 7.93
  P2  mean across all markers = 9.38
  P3  mean across all markers = 7.72

Overall mean: 8.34


In [9]:
# Full set of aggregation functions
data = np.array([
    [4, 7, 2, 9],
    [3, 8, 5, 1],
    [6, 2, 9, 4],
])

funcs = [
    ("sum",  np.sum),
    ("mean", np.mean),
    ("std",  np.std),
    ("min",  np.min),
    ("max",  np.max),
]

print(f"{'Function':<8} {'axis=None':>12} {'axis=0 (per col)':>20} {'axis=1 (per row)':>20}")
print("-" * 65)
for name, fn in funcs:
    all_val  = fn(data)
    col_vals = fn(data, axis=0)
    row_vals = fn(data, axis=1)
    print(f"{name:<8} {all_val:>12.2f} {str(col_vals.round(2)):>20} {str(row_vals.round(2)):>20}")

# Cumulative functions
print(f"\ncumsum(axis=1) - running total per row:")
print(np.cumsum(data, axis=1))

Function    axis=None     axis=0 (per col)     axis=1 (per row)
-----------------------------------------------------------------
sum             60.00        [13 17 16 14]           [22 17 21]
mean             5.00 [4.33 5.67 5.33 4.67]     [5.5  4.25 5.25]
std              2.68 [1.25 2.62 2.87 3.3 ]     [2.69 2.59 2.59]
min              1.00            [3 2 2 1]              [2 1 2]
max              9.00            [6 8 9 9]              [9 8 9]

cumsum(axis=1) - running total per row:
[[ 4 11 13 22]
 [ 3 11 16 17]
 [ 6  8 17 21]]


## 4. Broadcasting rules

Broadcasting allows operations between arrays with different shapes. NumPy stretches the smaller array conceptually (without actually copying data) to match the larger one.

**Rules (applied left to right on dimensions):**
1. **`Padding`:** If arrays have different `ndim`, pad the smaller shape with 1s on the **left**
2. **`Stretching`:** `Dimensions` of size 1 are **stretched** to match the other array
3. **`Incompatibility Error`:** If `dimensions` of two arrays don't match and neither is 1 --> raises a `ValueError`

**Note** that we used the word **dimensions** exclusively, not shape. There is a big difference between shape and dimensions (1D/2D/3D) but also a close relation. An array having shape (3,) might have `elements` but dimension is 1D, and an array having shape (4,3) have 4 rows and 3 columns and a dimension of two. For an operation to occur two arrays must have `same dimension and same shape.`

`Dimensions` tell you `how many axes` or directions an `array has` (e.g., 1D, 2D, 3D), while `shape` tells you exactly `how many elements` are `along each of those axes`.

In [ ]:
# Scalar broadcasts to every element (simplest case)
a = np.array([1, 2, 3, 4, 5])
print(f"a + 10   = {a + 10}")     # scalar 10 --> [10,10,10,10,10]
print(f"a * 2.5  = {a * 2.5}")
print(f"a > 3    = {a > 3}")

a + 10   = [11 12 13 14 15]
a * 2.5  = [ 2.5  5.   7.5 10.  12.5]
a > 3    = [False False False  True  True]


### 4.1 **Rule 1:** Padding Smaller Dimensions on the Left
When two arrays have a different number of dimensions (ndim), NumPy automatically pads the smaller shape with 1s on the left side until both array shapes have equal length.

In [5]:
# A: 2D array of shape (3, 4)
A = np.ones((3, 4))

# B: 1D array of shape (4,)
B = np.array([10, 20, 30, 40])

# --- STEP 1: PADDING ON THE LEFT (Rule 1) ---
# A shape: (3, 4)
# B shape: (4,)   --> Padded on the left with 1 --> becomes (1, 4)
#
# Visually, B changes from 1D:
# [10, 20, 30, 40]
#
# To a 2D matrix with 1 row:
# [[10, 20, 30, 40]]

# --- STEP 2: CHECK TRAILING DIMENSIONS ---
# Compare shapes from right to left:
# Trailing axis (axis 1): A has 4, B has 4  --> MATCHED!!!
# Leading axis  (axis 0): A has 3, B has 1  --> B HAS a '1', SO STRETCH IT! (Rule 2)

# --- STEP 3: STRETCHING (Rule 2) ---
# B shape: (1, 4) --> Stretched down 3 rows --> becomes (3, 4):
# [[10, 20, 30, 40],
#  [10, 20, 30, 40],
#  [10, 20, 30, 40]]

result = A + B

print(result)
# Output shape: (3, 4)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[[11. 21. 31. 41.]
 [11. 21. 31. 41.]
 [11. 21. 31. 41.]]


### 4.2 **Rule 2:** Stretching Dimensions of Size 1
Arrays with a dimension size equal to 1 in a given axis (row or column) are conceptually stretched to match the size of the `corresponding` dimension (larger dimension) of the `other array`.

In [3]:
# A: Column vector of shape (3, 1)
A = np.array([[1], 
              [2], 
              [3]])

# B: Row vector of shape (1, 4)
B = np.array([[10, 20, 30, 40]])

# Step 1: A (3, 1) is stretched horizontally to (3, 4)
# Step 2: B (1, 4) is stretched vertically to (3, 4)
result = A + B

print(result)
# Output shape: (3, 4)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


### 4.3 **Rule 3:** Error on Incompatible Dimensions
If two dimensions do not match and neither of them is equal to 1, broadcasting fails and NumPy raises a `ValueError`.

In [ ]:
# A: 2D matrix of shape (3, 4)
A = np.ones((3, 4))

# B: 1D vector of shape (3,)
B = np.array([10, 20, 30])

# --- STEP 1: PADDING ON THE LEFT (Rule 1) ---
# A shape: (3, 4)
# B shape: (3,)   --> Padded on the left with 1 --> becomes (1, 3)
#
# Visually, B changes from 1D:
# [10, 20, 30]
#
# To a 2D matrix with 1 row:
# [[10, 20, 30]]

# --- STEP 2: CHECK TRAILING DIMENSIONS (Right-to-Left) ---
# Compare shapes from right to left:
# Trailing axis (axis 1): A has 4, B has 3  --> MISMATCH!
#
# Since 4 != 3 AND neither number is equal to 1,
# NumPy CANNOT stretch this dimension. It immediately fails! (Rule 3)

try:
    result = A + B
except ValueError as e:
    print("ValueError:", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


### Solution to Rule 3 (Incompatibility of Arrays)
If we want an array B (which has 3 values) and a `shape of (3,)` to add to an array A (which has 3 rows) and a `shape of (3,4)`, we must make its shape (3, 1) instead of (3,) so that the **Rule 2: stretching** could be applied to it as it completes the assumption of having 1 in shape which can be stretched.

When numpy can't solve the compatibility issue internally using Rule 1 and 2 and `raises an error` then we do `manual` the broadcasting using `np.newaxis`. `np.newaxis` is literally an alias for Python's `None`. 
When we use `np.newaxis` (or `None`), python inserts a dimension of size 1 into the array's `shape` metadata tuple i.e. (1,3) at that exact position.


#### Example

Suppose we have a 1D array:

```python
x = np.array([10, 20, 30]) 
# shape: (3,) | ndim: 1

```

If we apply `np.newaxis` in different positions:

1. **`x[np.newaxis, :]` (or `x[None, :]`)**
* **Action:** Inserts a new dimension at **axis 0** (left).
* **New Shape:** `(1, 3)` - A 2D array with 1 row and 3 columns.
* **Visual Layout:** `[[10, 20, 30]]`


2. **`x[:, np.newaxis]` (or `x[:, None]`)**
* **Action:** Inserts a new dimension at **axis 1** (right).
* **New Shape:** `(3, 1)` - A 2D array with 3 rows and 1 column.
* **Visual Layout:**
```python
[[10],
 [20],
 [30]]

```

### Why do we use it?

It explicitly places the dimension size `1` on the exact side (left or right) where NumPy's `broadcasting rules` need it to perform element-wise arithmetic.

In [ ]:
A = np.ones((3, 4))
B = np.array([10, 20, 30])

# Change B from shape (3,) to (3, 1) using np.newaxis
B_column = B[:, np.newaxis] 

# B_column shape is now (3, 1):
# [[10],
#  [20],
#  [30]]

# --- CHECK DIMENSIONS (Right-to-Left) ---
# Trailing axis (axis 1): A has 4, B_column has 1 --> B HAS A '1', SO STRETCH IT! (Rule 2)
# Leading axis  (axis 0): A has 3, B_column has 3 --> MATCH!

# --- STRETCHING (Rule 2) ---
# B_column shape: (3, 1) --> Stretched across 4 columns --> becomes (3, 4):
# [[10, 10, 10, 10],
#  [20, 20, 20, 20],
#  [30, 30, 30, 30]]

result = A + B_column
print(result)

### **Some More Examples of BroadCasting**

In [ ]:
# 2D array + 1D array - adds vector to each row
m = np.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
])
v = np.array([10, 20, 30])   # shape (3,)

print(f"Matrix m (3,3):\n{m}")
print(f"Vector v (3,):  {v}")
print(f"\nm + v (v added to each row):\n{m + v}")
# Broadcasting: v shape (3,) --> (1,3) --> (3,3)

# Subtract column mean from each column (centering)
col_means = m.mean(axis=0)      # shape (3,)
centered = m - col_means        # each row has col_means subtracted
print(f"\nColumn means: {col_means}")
print(f"Column-centered:\n{centered}")

Matrix m (3,3):
[[1 2 3]
 [4 5 6]
 [7 8 9]]
Vector v (3,):  [10 20 30]

m + v (v added to each row):
[[11 22 33]
 [14 25 36]
 [17 28 39]]

Column means: [4. 5. 6.]
Column-centered:
[[-3. -3. -3.]
 [ 0.  0.  0.]
 [ 3.  3.  3.]]


In [ ]:
# (3,1) and (1,4) broadcast to (3,4)
rows = np.array([
                [1],
                [2],
                [3]])    # shape (3,1)
cols = np.array([10, 20, 30, 40])  # shape (4,)  --> (1,4) internally

result = rows + cols  # shape (3,4)
print(f"rows (3,1):\n{rows}")
print(f"cols (1,4): {cols}")
print(f"\nresult = rows + cols --> shape {result.shape}:")
print(result)

# Practical: compute all pairwise distances (x_1 - x_2) between 4 values
x = np.array([0, 1, 3, 6])
dists = np.abs(x[:, np.newaxis] - x[np.newaxis, :])
print(f"\nPairwise distances:")
print(dists)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

rows (3,1):
[[1]
 [2]
 [3]]
cols (1,4): [10 20 30 40]

result = rows + cols --> shape (3, 4):
[[11 21 31 41]
 [12 22 32 42]
 [13 23 33 43]]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Pairwise distances:
[[0 1 3 6]
 [1 0 2 5]
 [3 2 0 3]
 [6 5 3 0]]


## 5. Matrix operations

Linear algebra operations that form the backbone of machine learning algorithms.

In [14]:
# Matrix multiplication - NOT element-wise
# Use @ operator (Python 3.5+) or np.dot()
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print(f"A:\n{A}")
print(f"B:\n{B}")
print(f"\nA * B (element-wise):\n{A * B}")    # NOT matrix multiply
print(f"\nA @ B (matrix multiply):\n{A @ B}")  # correct
print(f"np.dot(A,B):\n{np.dot(A, B)}")         # same as @

# Transpose
print(f"\nA.T (transpose):\n{A.T}")

A:
[[1 2]
 [3 4]]
B:
[[5 6]
 [7 8]]

A * B (element-wise):
[[ 5 12]
 [21 32]]

A @ B (matrix multiply):
[[19 22]
 [43 50]]
np.dot(A,B):
[[19 22]
 [43 50]]

A.T (transpose):
[[1 3]
 [2 4]]


### 5.1 How `np.linalg.norm()` Works
`np.linalg.norm()` calculates the **magnitude (length)** of a vector or matrix. By default, it computes the **Euclidean Norm** ($L_2$ norm).

#### Mathematical Formula for Vectors ($L_2$ Norm)
For a 1D vector $\vec{v} = [x_1, x_2, \dots, x_n]$, the formula is:

$$\Vert{}\vec{v}\Vert{}_2 = \sqrt{\sum_{i=1}^{n} \vert{}x_i\vert{}^2} = \sqrt{x_1^2 + x_2^2 + \dots + x_n^2}$$

### Matrix Norms & The `axis` Argument

When applied to a 2D matrix (like a table of gene expression levels or coordinates), passing `axis` tells NumPy along which axis to compute the length:

* **`np.linalg.norm(matrix, axis=0)`:** Computes the Euclidean norm for each **column** across rows.
* **`np.linalg.norm(matrix, axis=1)`:** Computes the Euclidean norm for each **row** across columns.
* **`keepdims=True`:** Preserves the collapsed dimension as a size of `1` (e.g., shape `(3, 1)` instead of `(3,)`), making it immediately compatible for normalizing data via broadcasting.


In [17]:
# Practical: Gene expression similarity
# Rows = genes, Cols = samples
expr = np.array([
    [2.1, 1.8, 3.2, 2.9],   # Gene A
    [0.8, 0.9, 0.7, 0.8],   # Gene B
    [3.5, 3.1, 3.8, 3.4],   # Gene C
])
print(f"Expression matrix shape: {expr.shape}  (3 genes x 4 samples)")

# Correlation matrix: genes x genes
# First normalize each gene (row) to unit length
norms    = np.linalg.norm(expr, axis=1, keepdims=True)
normed   = expr / norms
gene_cor = normed @ normed.T   # (3,4) @ (4,3) = (3,3) (Yields Cosine Similarity)

print(f"\nGene-gene cosine similarity:")
genes = ["GeneA", "GeneB", "GeneC"]
print(f"{'':>8}" + "".join(f"{g:>9}" for g in genes))
for i, row in enumerate(gene_cor):
    print(f"{genes[i]:>8}" + "".join(f"{v:>9.3f}" for v in row))

# Vector norm (Euclidean distance to origin)
v = np.array([3.0, 4.0])
print(f"\nNorm of [3, 4]: {np.linalg.norm(v)}")

# Manual Formula equivalent:
print(f"Manual norm: {np.sqrt(np.sum(v**2))}")

<IPython.core.display.Javascript object>

Expression matrix shape: (3, 4)  (3 genes x 4 samples)


<IPython.core.display.Javascript object>


Gene-gene cosine similarity:
            GeneA    GeneB    GeneC
   GeneA    1.000    0.954    0.985
   GeneB    0.954    1.000    0.987
   GeneC    0.985    0.987    1.000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Norm of [3, 4]: 5.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Manual norm: 5.0


#### Dot Product or Cosine Similarity (From 1st year Physics chapter 2) (**`for more discussion refer to the end of the Notebook`**)

* **`normed @ normed.T` Explanation:** In the practical gene section, you use cosine similarity:

$$\text{Cosine Similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

Dividing each row by its $L_2$ norm turns dot products into cosine similarities.


### Understanding the Euclidean Norm & Distance

#### 1. `Distance Between Two Points` vs. `Length of a Single Vector`

* **Distance Between Two Points:**

$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

Calculates how far apart Point 1 and Point 2 are in space.

* **Magnitude (Length) of a Single Vector:**

$$\Vert{}\vec{v}\Vert{} = \sqrt{x^2 + y^2}$$

Calculates how long a vector is from the **origin $(0,0)$** to the point $(x,y)$.

#### How They Are the Same Concept

If Point 1 is at the origin $(x_1, y_1) = (0, 0)$ and Point 2 is at $(x_2, y_2) = (x, y)$, substituting $(0,0)$ into the distance formula gives:

$$d = \sqrt{(x - 0)^2 + (y - 0)^2} = \sqrt{x^2 + y^2}$$

The vector magnitude formula is simply the Euclidean distance formula where Point 1 is anchored at the origin $(0,0)$.

#### Finding Distance Between Two Points in NumPy

To compute the distance between two distinct points, subtract one point from the other to get the displacement vector, then compute `np.linalg.norm()`:

In [18]:
p1 = np.array([1, 2])
p2 = np.array([4, 6])

# Option 1: Using the point-to-point formula manually
d_manual = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
print(f"Manual distance: {d_manual}")

# Option 2: Taking the norm of the difference vector (p2 - p1)
d_norm = np.linalg.norm(p2 - p1)
print(f"Norm distance: {d_norm}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Manual distance: 5.0


<IPython.core.display.Javascript object>

Norm distance: 5.0


### Step-by-Step Matrix Example: Column-wise vs. Row-wise Norms

Consider a matrix $M$ with 3 rows and 2 columns:

$$M = \begin{bmatrix} 3 & 1 \\ 4 & 2 \\ 0 & 2 \end{bmatrix}$$

In [19]:
M = np.array([[3.0, 1.0], 
              [4.0, 2.0], 
              [0.0, 2.0]])

<IPython.core.display.Javascript object>

#### 1. Column-wise Norm (`axis=0`)

* **What it does:** Calculates the length of each column down the rows.
* **Shape Transformation:** Collapses rows, converting shape `(3, 2)` $\rightarrow$ `(2,)`.
* **Math Calculations:**
* **Column 0:** $[3, 4, 0] \implies \sqrt{3^2 + 4^2 + 0^2} = \sqrt{25} = \mathbf{5.0}$
* **Column 1:** $[1, 2, 2] \implies \sqrt{1^2 + 2^2 + 2^2} = \sqrt{9} = \mathbf{3.0}$

In [ ]:
col_norms = np.linalg.norm(M, axis=0)
print(col_norms)

<IPython.core.display.Javascript object>

[5. 3.]


#### 2. Row-wise Norm (`axis=1`)

* **What it does:** Calculates the length of each row across the columns.
* **Shape Transformation:** Collapses columns, converting shape `(3, 2)` $\rightarrow$ `(3,)`.
* **Math Calculations:**
* **Row 0:** $[3, 1] \implies \sqrt{3^2 + 1^2} = \sqrt{10} \approx \mathbf{3.162}$
* **Row 1:** $[4, 2] \implies \sqrt{4^2 + 2^2} = \sqrt{20} \approx \mathbf{4.472}$
* **Row 2:** $[0, 2] \implies \sqrt{0^2 + 2^2} = \sqrt{4} = \mathbf{2.0}$

In [ ]:
row_norms = np.linalg.norm(M, axis=1)
print(row_norms.round(3))

<IPython.core.display.Javascript object>

[3.162 4.472 2.   ]


#### Tip: Preserving Dimensions for Broadcasting (`keepdims=True`)

To divide a matrix by its row norms (e.g., to normalize each row to unit length), pass `keepdims=True`:

In [22]:
row_norms_2d = np.linalg.norm(M, axis=1, keepdims=True)

print(row_norms_2d.shape)
# Output shape: (3, 1) <-- Ready to broadcast across the (3, 2) matrix!

# Unit length matrix normalization:
M_normalized = M / row_norms_2d

<IPython.core.display.Javascript object>

(3, 1)


### 5.2 Advanced Linear Algebra for ML (`np.linalg`)
Real-world machine learning algorithms (like PCA or Linear Regression equations) require extracting structural properties from matrices, finding maximum probability indices, and solving systems of equations.

- `np.argmax()` / `np.argmin()` --> returns the **index** of the maximum/minimum value. Essential for classification network outputs.
- `np.linalg.inv()` --> calculates the inverse of a matrix.
- `np.linalg.eig()` --> extracts Eigenvalues and Eigenvectors (the backbone of dimensionality reduction).

In [ ]:
# 1. Argmax in Machine Learning (Finding the predicted class)
# Imagine a neural network outputs probability scores for 3 classes (Cat, Dog, Bird) across 4 samples
probabilities = np.array([
    [0.15, 0.75, 0.10],  # Sample 1 -> Highest at index 1 (Dog)
    [0.80, 0.10, 0.10],  # Sample 2 -> Highest at index 0 (Cat)
    [0.05, 0.05, 0.90],  # Sample 3 -> Highest at index 2 (Bird)
])
predictions = np.argmax(probabilities, axis=1)
print(f"Highest probability indices per sample: {predictions}")

# 2. Matrix Inverse & Eigenvalues
X = np.array([[2, 1], 
              [1, 3]])

X_inverse = np.linalg.inv(X)
eigenvalues, eigenvectors = np.linalg.eig(X)

print(f"\nMatrix Inverse:\n{X_inverse}")
print(f"\nEigenvalues:   {eigenvalues.round(4)}")

## 6. Reshaping and stacking

In [6]:
flat = np.arange(24)
print(f"flat:   {flat}")

# reshape - change shape, same data (returns a VIEW)
m4x6 = flat.reshape(4, 6)
m2x3x4 = flat.reshape(2, 3, 4)   # 3D array
auto = flat.reshape(3, -1)        # -1: infer this dimension

print(f"\n(4,6):\n{m4x6}")
print(f"\n(3,-1) = (3,8):\n{auto}")

<IPython.core.display.Javascript object>

flat:   [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]

(4,6):
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]]

(3,-1) = (3,8):
[[ 0  1  2  3  4  5  6  7]
 [ 8  9 10 11 12 13 14 15]
 [16 17 18 19 20 21 22 23]]


### 6.1 Functions of `.flatten()` and `.ravel()`

Both `.flatten()` and `.ravel()` collapse a multi-dimensional array (e.g., 2D or 3D matrix) into a 1-dimensional array.

The critical difference lies in **memory management**:

| Feature | `.ravel()` | `.flatten()` |
| --- | --- | --- |
| **Return Type** | **Memory View** (whenever possible)| **Hard Copy** (always)|
| **Shares Memory?** | **Yes** - shares memory with the original array.| **No** - allocates brand new RAM memory.|
| **Modifications** | Modifying a raveled array **changes the original array**. | Modifying a flattened array **does not affect the original**. |
| **Speed / Overhead** | Faster & zero extra RAM usage. | Slower & uses double the memory. |


#### Rule of Thumb

* Use **`.ravel()`** when processing large data arrays in pipelines where speed and RAM efficiency matter.
* Use **`.flatten()`** when you need a completely isolated copy of the data without risk of altering original datasets.

In [7]:
# ravel vs flatten
m = np.array([[1,2,3],[4,5,6]])
print(f"\nravel  (view):   {m.ravel()}")
print(f"flatten (copy):  {m.flatten()}")
print(f"Shares memory - ravel:   {np.shares_memory(m, m.ravel())}")
print(f"Shares memory - flatten: {np.shares_memory(m, m.flatten())}")

<IPython.core.display.Javascript object>


ravel  (view):   [1 2 3 4 5 6]
flatten (copy):  [1 2 3 4 5 6]


<IPython.core.display.Javascript object>

Shares memory - ravel:   True


<IPython.core.display.Javascript object>

Shares memory - flatten: False


In [13]:
original = np.array([[1, 2], 
                     [3, 4]])
# 1. Using flatten()
flat = original.flatten()
flat[1] = 888  # Modify the second element

print(f'ORIGINAL UNTOUCHED! \n{original}')
print(f'\nFlatted array is: \n{flat}')

# 2. Using ravel()
rav = original.ravel()
rav[0] = 999  # Modify the first element


print(f'\nFlatted array with ravel()! \n{rav}')
print(f'\nORIGINAL CHANGED! \n{original}')

<IPython.core.display.Javascript object>

ORIGINAL UNTOUCHED! 
[[1 2]
 [3 4]]

Flatted array is: 
[  1 888   3   4]

Flatted array with ravel()! 
[999   2   3   4]

ORIGINAL CHANGED! 
[[999   2]
 [  3   4]]


### 6.2 Stacking arrays

In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])
b = np.array([[7, 8, 9],
              [10,11,12]])

# vstack - vertical (add rows)
v = np.vstack([a, b])
print(f"vstack shape: {v.shape}\n{v}")

# hstack - horizontal (add columns)
h = np.hstack([a, b])
print(f"\nhstack shape: {h.shape}\n{h}")

# Practical: combine patient batches
batch1 = np.random.randint(60, 100, (10, 3))   # 10 patients, 3 scores
batch2 = np.random.randint(60, 100, (15, 3))   # 15 patients, 3 scores
combined = np.vstack([batch1, batch2])
print(f"\nCombined batches: {batch1.shape} + {batch2.shape} = {combined.shape}")

vstack shape: (4, 3)
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]

hstack shape: (2, 6)
[[ 1  2  3  7  8  9]
 [ 4  5  6 10 11 12]]

Combined batches: (10, 3) + (15, 3) = (25, 3)


### 6.3 Array Splitting
Just as you can stack batches of data together, you frequently need to split matrices apart—most commonly when separating features ($X$) from target labels ($y$), or dividing data into train/test splits.

In [39]:
# Splitting a dataset horizontally (separating features from the last label column)
# 4 patients, 3 columns: [Age, HeartRate, HasDisease]
dataset = np.array([
    [25, 72, 0],
    [47, 85, 1],
    [31, 64, 0],
    [55, 90, 1]
])

# np.hsplit(matrix, [index]) splits columns at the specified boundary
X, y = np.hsplit(dataset, [2])
print("Features (X):")
print(X)
print("\nLabels (y):")
print(y.flatten())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Features (X):
[[25 72]
 [47 85]
 [31 64]
 [55 90]]

Labels (y):
[0 1 0 1]


## Practical

### Practical 1 - The axis parameter: which way does it collapse?
**Context:** A student computes the mean of a patient×marker matrix. They want the average reading per patient but accidentally compute the average across patients. Show the mental model clearly.

In [23]:
# Drug trial: 4 patients, 5 blood glucose readings over a week
readings = np.array([
    [5.1, 5.4, 5.3, 5.6, 5.2],   # P1 - stable
    [9.4, 8.8, 9.1, 9.7, 8.9],   # P2 - elevated
    [4.9, 5.0, 5.1, 4.8, 5.2],   # P3 - normal
    [11.3, 10.8, 11.7, 10.5, 11.1], # P4 - diabetic
])
print(f"Shape: {readings.shape}  ({readings.shape[0]} patients, {readings.shape[1]} readings)")

# axis=0: collapse patients --> mean per TIME POINT
mean_per_day = readings.mean(axis=0)
print(f"axis=0 (mean per day, across patients): {mean_per_day.round(2)}")
print(f"  --> answers: 'What was the average glucose for ALL patients on day 3?'")

# axis=1: collapse time --> mean per PATIENT
mean_per_patient = readings.mean(axis=1)
print(f"axis=1 (mean per patient, across days): {mean_per_patient.round(2)}")
print(f"  --> answers: 'What was patient 2's average glucose for the week?'")

# Mental model: axis=0 collapses DOWN the rows; axis=1 collapses ACROSS the columns

<IPython.core.display.Javascript object>

Shape: (4, 5)  (4 patients, 5 readings)
axis=0 (mean per day, across patients): [7.68 7.5  7.8  7.65 7.6 ]
  --> answers: 'What was the average glucose for ALL patients on day 3?'
axis=1 (mean per patient, across days): [ 5.32  9.18  5.   11.08]
  --> answers: 'What was patient 2's average glucose for the week?'


### Practical 2 - Broadcasting shapes: what works and what doesn't?
**Context:** Students encounter a `ValueError: operands could not be broadcast together` and don't know why. Walk through the rules.

In [24]:
# Show which shape combinations work
def try_broadcast(a_shape, b_shape):
    a = np.ones(a_shape)
    b = np.ones(b_shape)
    try:
        result = a + b
        print(f"  {str(a_shape):>10} + {str(b_shape):<10} --> {result.shape}  OK")
    except ValueError as e:
        print(f"  {str(a_shape):>10} + {str(b_shape):<10} --> ERROR: {e}")

print("Broadcasting compatibility table:")
pairs = [
    ((3,),    (3,)),       # same --> OK
    ((3, 4),  (4,)),       # (4,) treated as (1,4) --> broadcasts across rows --> OK
    ((3, 4),  (3, 1)),     # (3,1) broadcasts across columns --> OK
    ((3, 1),  (1, 4)),     # both broadcast --> (3,4) --> OK
    ((3, 4),  (3,)),       # (3,) treated as (1,3) --> doesn't match (,4) --> ERROR
    ((3, 4),  (4, 3)),     # completely different shapes --> ERROR
    ((2,3,4), (3, 4)),     # (3,4) --> (1,3,4) --> OK
]
for a, b in pairs:
    try_broadcast(a, b)

Broadcasting compatibility table:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

        (3,) + (3,)       --> (3,)  OK


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

      (3, 4) + (4, 3)     --> ERROR: operands could not be broadcast together with shapes (3,4) (4,3) 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   (2, 3, 4) + (3, 4)     --> (2, 3, 4)  OK


### Practical 3 - `*` is not matrix multiplication
**Context:** A student implements a neural network layer: `output = weights * input_vec`. The shapes match element-wise but the computation is wrong. This is the most common NumPy mistake in ML code.

In [25]:
# Simulating one layer: 3 neurons, 4 inputs
weights = np.array([[0.1, 0.2, 0.3, 0.4],
                    [0.5, 0.1, 0.6, 0.2],
                    [0.3, 0.7, 0.1, 0.5]])  # shape (3, 4)
inputs  = np.array([1.0, 2.0, 3.0, 4.0])   # shape (4,)

# WRONG: element-wise with broadcasting
try:
    wrong = weights * inputs    # (3,4) * (4,) --> element-wise! shape (3,4)
    print(f"weights * inputs shape:  {wrong.shape}   <-- WRONG (still 2D!)")
    print(f"Result:\n{wrong}")
except Exception as e:
    print(f"Error: {e}")

# CORRECT: matrix multiplication
correct = weights @ inputs    # (3,4) @ (4,) --> (3,)  one value per neuron
print(f"weights @ inputs shape:  {correct.shape}   <-- CORRECT (one output per neuron)")
print(f"Layer output: {correct.round(3)}")

# The rule: @ or np.dot() for matrix multiply. * is ALWAYS element-wise.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

weights * inputs shape:  (3, 4)   <-- WRONG (still 2D!)
Result:
[[0.1 0.4 0.9 1.6]
 [0.5 0.2 1.8 0.8]
 [0.3 1.4 0.3 2. ]]
weights @ inputs shape:  (3,)   <-- CORRECT (one output per neuron)
Layer output: [3.  3.3 4. ]


### Practical 4 - Centering and scaling the right way
**Context:** Before any ML algorithm, you need to normalise features/columns. Show how broadcasting makes this elegant vs looping.

In [26]:
# 5 patients, 4 clinical features: [age, glucose, bp_sys, bmi]
data = np.array([
    [34, 5.1, 120, 22.4],
    [51, 9.4, 145, 28.7],
    [28, 4.8, 118, 20.1],
    [67, 11.3, 155, 31.2],
    [45, 6.2, 130, 24.8],
])
features = ["Age", "Glucose", "BP_sys", "BMI"]

print("Raw data:")
print(data)

# Step 1: compute mean and std per feature (axis=0 = per column)
means = data.mean(axis=0)
stds  = data.std(axis=0)

print(f"Mean per feature: {means.round(2)}")
print(f"Std  per feature: {stds.round(2)}")

# Step 2: z-score normalise - broadcasting does the right thing
# (5,4) - (4,) --> (4,) broadcasts to (5,4): subtract mean from each row
# (5,4) / (4,) --> same pattern for std 
z_scored = (data - means) / stds

print(f"Z-scored data (mean≈0, std≈1 per column):")
print(z_scored.round(3))
print(f"Verification - column means: {z_scored.mean(axis=0).round(10)}")
print(f"             - column stds:  {z_scored.std(axis=0).round(4)}")

<IPython.core.display.Javascript object>

Raw data:
[[ 34.    5.1 120.   22.4]
 [ 51.    9.4 145.   28.7]
 [ 28.    4.8 118.   20.1]
 [ 67.   11.3 155.   31.2]
 [ 45.    6.2 130.   24.8]]
Mean per feature: [ 45.     7.36 133.6   25.44]
Std  per feature: [13.64  2.56 14.35  4.05]
Z-scored data (mean≈0, std≈1 per column):
[[-0.807 -0.884 -0.948 -0.751]
 [ 0.44   0.798  0.795  0.805]
 [-1.247 -1.001 -1.087 -1.319]
 [ 1.613  1.541  1.492  1.423]
 [ 0.    -0.454 -0.251 -0.158]]
Verification - column means: [ 0. -0.  0.  0.]
             - column stds:  [1. 1. 1. 1.]


## 8. Practical - Vectorized BMI & Z-score Calculator
**Difficulty: Medium**

In [27]:
# 50 simulated patient records: weight (kg), height (m)
np.random.seed(42)
weight = np.random.normal(70, 15, 50).clip(40, 130)   # kg
height = np.random.normal(1.70, 0.12, 50).clip(1.4, 2.1)  # m

# BMI: vectorized - no loop
bmi = weight / (height ** 2)
print(f"BMI array shape: {bmi.shape}")
print(f"BMI range: [{bmi.min():.1f}, {bmi.max():.1f}]")
print(f"Mean BMI:  {bmi.mean():.2f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

BMI array shape: (50,)
BMI range: [13.4, 36.6]
Mean BMI:  23.18


In [28]:
np.random.seed(42)
weight = np.random.normal(70, 15, 50).clip(40, 130)
height = np.random.normal(1.70, 0.12, 50).clip(1.4, 2.1)
bmi    = weight / (height ** 2)

# Classify using np.select (vectorized multi-condition)
conditions = [bmi < 18.5, bmi < 25.0, bmi < 30.0]
choices    = ["Underweight", "Normal", "Overweight"]
categories = np.select(conditions, choices, default="Obese")

# Count each category
for cat in ["Underweight", "Normal", "Overweight", "Obese"]:
    count = (categories == cat).sum()
    pct   = count / len(categories) * 100
    bar   = "█" * count
    print(f"  {cat:<12} {count:3d} ({pct:.0f}%)  {bar}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  Underweight    8 (16%)  ████████
  Normal        25 (50%)  █████████████████████████
  Overweight    10 (20%)  ██████████
  Obese          7 (14%)  ███████


In [29]:
# Exam scores for 30 students, 5 subjects
np.random.seed(42)
scores = np.random.randint(40, 101, (30, 5)).astype(float)

# Z-score normalisation per subject (axis=0 = per column)
means    = scores.mean(axis=0)
stds     = scores.std(axis=0)
z_scores = (scores - means) / stds

print(f"Scores shape:   {scores.shape}")
print(f"Z-scores shape: {z_scores.shape}")
print(f"Per-subject means (original): {means.round(1)}")
print(f"Per-subject means (z-scored): {z_scores.mean(axis=0).round(10)}")
print(f"Per-subject stds  (z-scored): {z_scores.std(axis=0).round(4)}")

# Identify students with z-score < -1.5 in any subject (at risk)
at_risk = (z_scores < -1.5).any(axis=1)
print(f"At-risk students (z < -1.5 in any subject): {at_risk.sum()}")
print(f"Their student indices: {np.where(at_risk)[0]}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Scores shape:   (30, 5)
Z-scores shape: (30, 5)
Per-subject means (original): [72.8 70.8 71.1 67.6 68.9]
Per-subject means (z-scored): [ 0.  0. -0.  0.  0.]
Per-subject stds  (z-scored): [1. 1. 1. 1. 1.]
At-risk students (z < -1.5 in any subject): 7


<IPython.core.display.Javascript object>

Their student indices: [ 4  5 10 13 17 25 26]


In [30]:
# Verify a 3x3 matrix multiplication manually
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
B = np.array([[9, 8, 7],
              [6, 5, 4],
              [3, 2, 1]])

C = A @ B
print(f"A @ B ={C}")

# Verify element (0,0) manually: row 0 of A dot col 0 of B
manual_00 = A[0, :] @ B[:, 0]
print(f"Manual C[0,0]: row0(A)·col0(B) = {A[0,:]} · {B[:,0]} = {manual_00}")
print(f"NumPy  C[0,0]: {C[0,0]}")
print(f"Match: {manual_00 == C[0,0]}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

A @ B =[[ 30  24  18]
 [ 84  69  54]
 [138 114  90]]
Manual C[0,0]: row0(A)·col0(B) = [1 2 3] · [9 6 3] = 30
NumPy  C[0,0]: 30
Match: True


## Summary

| Concept | Key point |
|---|---|
| Element-wise ops | `+`, `-`, `*`, `/` between same-shape arrays --> pairwise operation |
| `axis=0` | Collapse **rows** --> result has shape `(cols,)` --> one value per column |
| `axis=1` | Collapse **columns** --> result has shape `(rows,)` --> one value per row |
| Broadcasting rule | Dimensions of size 1 stretch to match; trailing dims must align |
| `@` vs `*` | `@` is matrix multiply; `*` is element-wise. Never confuse them |
| `np.clip(a, lo, hi)` | Cap values between lo and hi - cleaner than manual boolean masks |
| `np.select(conds, choices)` | Vectorized if/elif/else - replaces loops for classification |
| `.reshape(r, -1)` | `-1` lets NumPy infer the missing dimension |

**Before Sunday:** A matrix A has shape (4, 3). A vector v has shape (3,). What is the shape of `A @ v`? What is the shape of `A.T @ A`?

## **Dot Product or Cosine Similarity** (`Optional`)

At its core, **Cosine Similarity** measures the **direction or angle between two vectors**, ignoring how big (long) they are.
Instead of measuring the distance between two points in space, it asks: **"Are these two vectors pointing in roughly the same direction?"**

## 1. The Analogy & Intuition

Imagine two people rating movies from 1 to 5 stars:

* **User A** gives *Sci-Fi Movie X* a **5** and *Action Movie Y* a **4**.
* **User B** gives *Sci-Fi Movie X* a **2.5** and *Action Movie Y* a **2**.

If you calculate Euclidean distance, User A and User B seem far apart because User A gives high ratings overall and User B gives low ratings overall.

However, **their taste ratio is identical (5:4 vs 2.5:2)**! They both prefer Sci-Fi over Action by the exact same proportion. Cosine similarity catches this—it recognizes that their preference vectors are pointing in the exact same direction.

## 2. The Math Behind It

The formula computes the cosine of the angle $\theta$ between two vectors, $A$ and $B$:

$$\text{Cosine Similarity} = \cos(\theta) = \frac{A \cdot B}{\Vert{}A\Vert{}_2 \Vert{}B\Vert{}_2}$$

* **$A \cdot B$ (Dot Product):** Multiplies corresponding elements and adds them up.
* **$\Vert{}A\Vert{}_2 \Vert{}B\Vert{}_2$ (Product of $L_2$ Norms):** Divides by their individual lengths so that scale/magnitude doesn't bias the result.

### What the Values Mean:

| Score | Angle ($\theta$) | Meaning | Example |
| --- | --- | --- | --- |
| **$+1$** | $0^\circ$ | **Perfectly Identical** direction | Preference ratios match completely |
| **$0$** | $90^\circ$ | **Orthogonal** (Unrelated) | Share zero traits/words in common |
| **$-1$** | $180^\circ$ | **Opposite** direction | Diametrically opposed tastes/signals |

> **Note on Text & Non-negative Data:** In fields like text mining, gene expression, or recommender systems, values are usually non-negative ($0$ to $\infty$). Thus, cosine similarity usually ranges strictly between **$0.0$ and $1.0$**.

## 3. Why & Where Is It Used?

Cosine similarity shines when **magnitude varies, but proportional composition matters**.

### A. Natural Language Processing (NLP) & Search Engines

* **Problem:** Short articles and long books might discuss the exact same topic (e.g., "artificial intelligence"), but the book mentions the words 1,000 times while the article mentions them 5 times.
* **Solution:** Euclidean distance would say the book and article are completely different because of word count. Cosine similarity ignores text length and compares the *proportion* of words used.

### B. Bioinformatics & Genomics

* **Problem:** Comparing gene expression levels across samples where one sample might simply have higher overall sequencing depth or technical scaling.
* **Solution:** Normalizing gene vectors to unit length and taking their dot product (yielding cosine similarity) reveals which genes co-regulate or behave similarly across samples regardless of overall concentration differences.

### C. Recommendation Systems

* **Problem:** Streaming services (e.g., Spotify, Netflix) want to recommend content based on viewing patterns.
* **Solution:** If two users watch the same genres in similar proportions—even if one watches 50 hours a week and the other watches 5 hours—their cosine similarity will be near $1.0$.


In [ ]:
# Vector A and Vector B
a = np.array([2.1, 1.8, 3.2])
b = np.array([4.2, 3.6, 6.4])  # Exactly twice the vector A!

# Method 1: Using the formula directly
dot_product = np.dot(a, b)
norm_a = np.linalg.norm(a)
norm_b = np.linalg.norm(b)

cos_sim = dot_product / (norm_a * norm_b)
print(f"Cosine Similarity: {cos_sim:.4f}")

# Method 2: Normalize vectors first (unit length), then take dot product
a_normed = a / norm_a
b_normed = b / norm_b

cos_sim_fast = a_normed @ b_normed
print(f"Cosine Similarity (normalized @): {cos_sim_fast:.4f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cosine Similarity: 1.0000
Cosine Similarity (normalized @): 1.0000


### 1. NLP / Document Similarity

Imagine we want to compare how similar two documents are based on how often three specific words appear: `["python", "data", "code"]`.

* **Document 1 (Short Tweet):** `[2, 1, 0]` (2 times "python", 1 time "data")
* **Document 2 (Long Article):** `[10, 5, 0]` (10 times "python", 5 times "data")

Notice that Document 2 has **5 times more words**, but the *proportion* of words is identical ($2:1$).

#### Manual Math

1. **Dot Product ($A \cdot B$):**

$$A \cdot B = (A_x \times B_x) + (A_y \times B_y) + (A_z \times B_z)$$
$$(2 \times 10) + (1 \times 5) + (0 \times 0) = 20 + 5 + 0 = \mathbf{25}$$

1. **Magnitudes ($\Vert{}A\Vert{}_2$ and $\Vert{}B\Vert{}_2$):**

$$\Vert{}A\Vert{}_2 = \sqrt{2^2 + 1^2 + 0^2} = \sqrt{4 + 1 + 0} = \sqrt{5} \approx \mathbf{2.236}$$

$$\Vert{}B\Vert{}_2 = \sqrt{10^2 + 5^2 + 0^2} = \sqrt{100 + 25 + 0} = \sqrt{125} \approx \mathbf{11.180}$$

3. **Cosine Similarity:**

$$\text{Cosine Similarity} = \frac{25}{2.236 \times 11.180} = \frac{25}{25} = \mathbf{1.0}$$

> **Takeaway:** Even though the word counts are completely different in scale, cosine similarity correctly identifies that both documents have **identical word distribution** ($1.0$).


### 2. Recommender System (Movie Ratings)

Suppose we want to find out if **User 1** and **User 2** have similar movie tastes based on their ratings (out of 5 stars) across 3 genres: `[Sci-Fi, Action, Drama]`.

* **User 1 (Generous Rater):** `[5.0, 4.0, 1.0]` (Loves Sci-Fi & Action, dislikes Drama)
* **User 2 (Harsh Rater):** `[2.5, 2.0, 0.5]` (Same relative preferences, but gives lower scores)
* **User 3 (Drama Fan):** `[1.0, 1.0, 5.0]` (Opposite tastes)

In [36]:
# Rows = Users, Columns = Genres
ratings = np.array([
    [5.0, 4.0, 1.0],  # User 1
    [2.5, 2.0, 0.5],  # User 2
    [1.0, 1.0, 5.0]   # User 3
])

# Step 1: Compute row norms (keepdims=True maintains (3, 1) shape)
norms = np.linalg.norm(ratings, axis=1, keepdims=True)

# Step 2: Scale matrix to unit vectors
normed_ratings = ratings / norms

# Step 3: Compute User-by-User Cosine Similarity via Matrix Multiplication
user_similarity = normed_ratings @ normed_ratings.T

print("User Similarity Matrix:")
print(user_similarity.round(3))

a = ''' 
Takeaway: User 1 and User 2 get a similarity score of 1.000 because their rating ratios 
across genres match perfectly, whereas User 3 gets a low similarity score of 0.375.
'''

print(a)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

User Similarity Matrix:
[[1.    1.    0.416]
 [1.    1.    0.416]
 [0.416 0.416 1.   ]]
 
Takeaway: User 1 and User 2 get a similarity score of 1.000 because their rating ratios 
across genres match perfectly, whereas User 3 gets a low similarity score of 0.375.

